# Authenticated Drive inventory
List metadata only for the supplied data and model folders using your own Google account. Mounted paths and translation filenames below match your saved inventory. This does not read file contents, download weights, make files public, or grant other users access. Review notebook outputs before sharing because filenames can be sensitive. Only standard Colab Google authorization and Drive mount dialogs are used.
Data: [139FuEvqxcZzVglh0qNBIIDaXusjpgFuh](https://drive.google.com/drive/folders/139FuEvqxcZzVglh0qNBIIDaXusjpgFuh?usp=drive_link). Models: [13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6](https://drive.google.com/drive/folders/13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6?usp=drive_link).

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, project = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
DATA_FOLDER_ID = '139FuEvqxcZzVglh0qNBIIDaXusjpgFuh'
MODEL_FOLDER_ID = '13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6'


In [ ]:
import json
from googleapiclient.errors import HttpError

def inventory(folder_id):
    fields = 'id,name,mimeType,webViewLink,shortcutDetails(targetId,targetMimeType)'
    metadata = drive_api.files().get(fileId=folder_id, fields=fields, supportsAllDrives=True).execute()
    print('Folder metadata:', json.dumps(metadata, indent=2))
    if metadata['mimeType'] != 'application/vnd.google-apps.folder':
        raise ValueError('Expected a folder. Inspect shortcut metadata and choose the actual target ID explicitly.')
    page_token = None
    while True:
        page = drive_api.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            spaces='drive', pageSize=100, pageToken=page_token,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
            fields='nextPageToken,files(id,name,mimeType,size,modifiedTime,shortcutDetails(targetId,targetMimeType))',
        ).execute()
        for item in page.get('files', []):
            print(json.dumps(item))
        page_token = page.get('nextPageToken')
        if not page_token:
            break

for role, folder_id in [('DATA', DATA_FOLDER_ID), ('MODELS', MODEL_FOLDER_ID)]:
    print(role, folder_id)
    try:
        inventory(folder_id)
    except HttpError as error:
        print('Metadata request failed with status', error.resp.status,
              '- check account access, folder ID and Drive API availability. No public download fallback is used.')


## Use your mounted data and model paths
Your saved inventory identifies the data folder as `data` and the model folder as `MT`. The paths below use `/content/drive/MyDrive/data` and `/content/drive/MyDrive/MT`, with the exact translation filenames recorded in that inventory. Twi filenames differ by split: `train_twi`, `val_tw` and `test_twi`. The selected checkpoint is `/content/drive/MyDrive/MT/OPUS-mt-tw-fr-tuned`. The check lists paths only; it does not read corpus contents or load model weights. File presence does not prove sentence alignment or a model's language direction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/data')
DRIVE_ROOT = Path('/content/drive/MyDrive/MT')
MODEL_ROOT = DRIVE_ROOT
CHECKPOINT_PATH = MODEL_ROOT / 'OPUS-mt-tw-fr-tuned'
DATA_FILES = {
    'train': {'en': DATA_ROOT / 'train_en', 'tw': DATA_ROOT / 'train_twi', 'fr': DATA_ROOT / 'train_fr'},
    'validation': {'en': DATA_ROOT / 'val_en', 'tw': DATA_ROOT / 'val_tw', 'fr': DATA_ROOT / 'val_fr'},
    'test': {'en': DATA_ROOT / 'test_en', 'tw': DATA_ROOT / 'test_twi', 'fr': DATA_ROOT / 'test_fr'},
}

missing = []
for role, root in [('data', DATA_ROOT), ('models', MODEL_ROOT), ('checkpoint', CHECKPOINT_PATH)]:
    exists = root.is_dir()
    print(f'{role}: {root} | directory exists: {exists}')
    if not exists:
        missing.append(str(root))
for split, languages in DATA_FILES.items():
    for language, path in languages.items():
        exists = path.is_file()
        print(f'{split}/{language}: {path} | file exists: {exists}')
        if not exists:
            missing.append(str(path))
if missing:
    raise FileNotFoundError('Missing mounted Drive paths:\n' + '\n'.join(missing))


Transfer the confirmed paths and actual filenames to `01_colab_train.ipynb`; transfer the model root to `02_colab_translate.ipynb`. Do not paste tokens into notebooks or share saved authorization outputs. Official references: [Drive files.get](https://developers.google.com/workspace/drive/api/reference/rest/v3/files/get), [Drive files.list](https://developers.google.com/workspace/drive/api/reference/rest/v3/files/list), [Colab Drive access and limitations](https://research.google.com/colaboratory/faq.html).